# Demo: flat vs. hierarchical classification of frog calls

Each section calls the part scripts in the repo root and prints their matrix rows, the same ones
`python partN_*.py` prints, with plots of the results. All models and metrics come from the
scripts. Running every cell takes about 90 s, most of it in bagging, boosting and the forest.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

import part1_decision_tree as p1
import part2_bagging_boosting as p2
import part3_random_forest as p3
import part4_hierarchical as p4
import part5_naive_bayes as p5
from common import CRITERIA, Data, accuracy, confusion_matrix, print_rows

# Plot colours: two series colours, a sequential ramp and a diverging pair.
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e1e0d9"
SEQUENTIAL = LinearSegmentedColormap.from_list(
    "blues", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)
DIVERGING = LinearSegmentedColormap.from_list(
    "blue-red", ["#184f95", "#6da7ec", "#f0efec", "#f0a3a2", "#e34948", "#9e2b2a"]
)
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": INK,
        "axes.titlesize": 11,
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "lines.linewidth": 2,
        "legend.frameon": False,
    }
)


def heatmap(ax, cm, labels, title):
    """A confusion matrix, rows true and columns predicted: the counts as text, each cell
    coloured by its share of the true class's rows, so small classes show as clearly as big ones."""
    share = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(share, cmap=SEQUENTIAL, vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(labels)), labels, fontsize=8)
    for i, j in np.ndindex(cm.shape):
        if cm[i, j]:
            dark = share[i, j] > 0.5
            ax.text(
                j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if dark else INK
            )
    ax.set(xlabel="predicted", ylabel="true")
    ax.set_title(title, loc="left")
    ax.grid(False)


data = Data()
X, y = data.X_train, data.y_train
print(f"train {len(y)} rows, test {len(data.y_test)} rows, {X.shape[1]} MFCC features")

## Part 1: the decision tree

Binary splits $x_j \le t$ chosen by the largest Gini decrease or information gain, with sample
weights. Every threshold of every feature is scored at once from cumulative class weights. The
handout's sanity checks run first.

In [ ]:
p1.sanity_checks()
trees = {c: p1.DecisionTree(c).fit(X, y) for c in CRITERIA}
print_rows([("1. Single Decision Tree", c.capitalize(), *data.scores(trees[c])) for c in CRITERIA])
for c, tree in trees.items():
    root = tree.nodes[0]
    print(
        f"{c}: depth {tree.depth}, {tree.n_leaves} leaves; root split MFCC {root['feature'] + 1}"
        f" <= {root['threshold']:.4f}, gain {root['gain']:.4f}"
    )

In [ ]:
depths = [1, 2, 3, 5, 8, 12, None]
sweep = {}
for c in CRITERIA:
    for depth in depths:
        tree = p1.DecisionTree(c, max_depth=depth).fit(X, y)
        sweep[c, depth] = (
            accuracy(y, tree.predict(X)),
            accuracy(data.y_test, tree.predict(data.X_test)),
        )

fig, ax = plt.subplots(figsize=(7.5, 3.8))
x = np.arange(len(depths))
for c, color in zip(CRITERIA, (BLUE, ORANGE)):
    train, test = (100 * np.array([sweep[c, d][i] for d in depths]) for i in (0, 1))
    ax.plot(x, train, color=color, linestyle="--", marker="o", markersize=5, label=f"{c}, train")
    ax.plot(x, test, color=color, marker="o", markersize=5, label=f"{c}, test")
ax.set_xticks(x, [str(d) for d in depths])
ax.set(xlabel="max_depth", ylabel="accuracy (%)")
ax.set_title("Training accuracy keeps rising, test accuracy levels off", loc="left")
ax.legend(ncol=2, loc="lower right")
plt.show()

A fitted tree is a list of nodes (`tree.nodes`), each with its feature (a 0-based column, so 12
is MFCC 13; -1 at a leaf), threshold, gain, majority class (`value`, an index into
`tree.classes_`), row count, class weights and depth. Here, a tree limited to depth 3 and to
nodes of at least 50 rows:

In [ ]:
tree = p1.DecisionTree("entropy", max_depth=3, min_samples_split=50).fit(X, y)
print(f"{tree.n_leaves} leaves, test {100 * accuracy(data.y_test, tree.predict(data.X_test)):.2f}%")
root = tree.nodes[0]
majority = tree.classes_[root["value"]]
if root["feature"] >= 0:
    print(f"root: MFCC {root['feature'] + 1} <= {root['threshold']:.4f}, majority {majority}")
else:  # feature -1: the root is a leaf
    print(f"root: a leaf, predicting {majority}")
root

## Part 2: bagging and AdaBoost (SAMME)

Bagging votes over 25 unconstrained trees grown on bootstrap samples, which cuts variance.
AdaBoost reweights the training rows over 50 rounds of depth-3 trees, which cuts bias; decision
stumps (depth 1) are run for comparison.

In [ ]:
bagging = {c: p2.Bagging(25, c).fit(X, y) for c in CRITERIA}
boosting = {c: p2.AdaBoostSAMME(50, c, max_depth=3).fit(X, y) for c in CRITERIA}
stumps = {c: p2.AdaBoostSAMME(50, c, max_depth=1).fit(X, y) for c in CRITERIA}
rows = [("2a. Bagging", c.capitalize(), *data.scores(bagging[c])) for c in CRITERIA]
rows += [("2b. AdaBoost", c.capitalize(), *data.scores(boosting[c])) for c in CRITERIA]
print_rows(rows)

In [ ]:
def staged_test_accuracy(model):
    """Test accuracy of the vote after each round: running sums of the alpha-weighted votes."""
    votes = np.array(
        [np.searchsorted(model.classes_, t.predict(data.X_test)) for t in model.trees_]
    )
    hits = votes[:, :, None] == np.arange(len(model.classes_))
    scores = np.cumsum(np.array(model.alphas_)[:, None, None] * hits, axis=0)
    truth = np.searchsorted(model.classes_, data.y_test)
    return 100 * (scores.argmax(axis=2) == truth).mean(axis=1)


for model in (*boosting.values(), *stumps.values()):  # the running vote ends at predict's answer
    final = 100 * accuracy(data.y_test, model.predict(data.X_test))
    assert np.isclose(staged_test_accuracy(model)[-1], final)

fig, axes = plt.subplots(2, 2, figsize=(11, 6.4), sharex=True)
for col, c in enumerate(CRITERIA):
    for model, color, label in ((boosting[c], BLUE, "depth 3"), (stumps[c], ORANGE, "stumps")):
        rounds = np.arange(1, len(model.errors_) + 1)  # the rounds kept
        axes[0, col].plot(rounds, staged_test_accuracy(model), color=color, label=label)
        axes[1, col].plot(rounds, model.errors_, color=color, label=label)
    axes[1, col].axhline(0.9, color=MUTED, linestyle=":", linewidth=1.5)
    edge = axes[1, col].get_yaxis_transform()  # x across the axes, y in data units
    axes[1, col].text(
        1, 0.91, "limit 1 - 1/K = 0.9", ha="right", va="bottom", color=MUTED, transform=edge
    )
    axes[0, col].set_title(f"{c}: test accuracy of the vote (%)", loc="left")
    axes[1, col].set_title(f"{c}: weighted error of each round", loc="left")
    axes[1, col].set(xlabel="boosting round", ylim=(0, 1))
    axes[0, col].legend(loc="lower right")
for ax in axes[0]:
    ax.set_ylim(40, 100)
plt.tight_layout()
plt.show()

A stump names at most two of the ten species, so its error climbs close to the 0.9 limit and
its say (alpha) shrinks. A deeper weak learner needs fewer rounds: 20 rounds of depth-5 trees.

In [ ]:
model = p2.AdaBoostSAMME(20, "gini", max_depth=5).fit(X, y)
test = 100 * accuracy(data.y_test, model.predict(data.X_test))
print(f"{len(model.trees_)} rounds, test {test:.2f}%")
print("alpha per round:", np.round(model.alphas_, 2))

## Part 3: the random forest

Bagging plus a fresh random subset of 4 of the 22 MFCCs at every split. The trees get weaker
but less alike, and the vote gets stronger.

In [ ]:
forest = {c: p3.RandomForest(50, c, 4).fit(X, y) for c in CRITERIA}
print_rows([("3. Random Forest", c.capitalize(), *data.scores(forest[c])) for c in CRITERIA])


def alike(model):
    """Mean pairwise agreement of the trees on the test set, and their mean test accuracy."""
    votes = model.votes(data.X_test)
    tree_acc = np.mean([accuracy(data.y_test, model.classes_[v]) for v in votes])
    return p3.agreement(votes, len(model.classes_)), 100 * tree_acc


stats = {
    (c, name): alike(m)
    for c in CRITERIA
    for name, m in (("bagging", bagging[c]), ("forest", forest[c]))
}
fig, axes = plt.subplots(1, 2, figsize=(11, 2.6))
for ax, k, title in (
    (axes[0], 0, "Pairwise agreement of the trees"),
    (axes[1], 1, "Mean single-tree test accuracy (%)"),
):
    for row, c in enumerate(CRITERIA):
        b, f = stats[c, "bagging"][k], stats[c, "forest"][k]
        ax.plot([b, f], [row, row], color=GRID, linewidth=4, zorder=1)
        ax.scatter([b], [row], s=70, color=BLUE, zorder=2, label="bagging" if row == 0 else None)
        ax.scatter(
            [f], [row], s=70, color=ORANGE, zorder=2, label="random forest" if row == 0 else None
        )
    ax.set_yticks([0, 1], list(CRITERIA))
    ax.set_ylim(-0.7, 2.1)
    ax.set_title(title, loc="left")
axes[0].legend(loc="upper left", ncol=2)
plt.tight_layout()
plt.show()

Feature subsets of m = 1, 4 and 22 per split, with 25 trees each (the matrix uses 50, so m = 4
gives 97.85% here against 97.78%). Fewer candidates make the trees less alike and individually
weaker. m = 22 is bagging as an algorithm but not the bagging row: the feature draws share the
random stream with the bootstraps, so the bootstraps differ after the first tree.

In [ ]:
for m in (1, 4, 22):
    model = p3.RandomForest(25, "gini", m).fit(X, y)
    agree, tree_acc = alike(model)
    test = 100 * accuracy(data.y_test, model.predict(data.X_test))
    print(f"m = {m:>2}: forest {test:.2f}%, agreement {agree:.3f}, mean tree {tree_acc:.2f}%")

## Part 4: the hierarchy

A router tree predicts the family, then that family's expert tree predicts the species. The
experts learn from the rows of their true family; the router's mistakes only matter at test time.
Each cell of the confusion matrices shows a count and is shaded by its share of the true class's
rows.

In [ ]:
hierarchy = {c: p4.fit_hierarchy(c, X, data.f_train, y) for c in CRITERIA}
train = {c: p4.analyse(hierarchy[c], X, data.f_train, y) for c in CRITERIA}
test = {c: p4.analyse(hierarchy[c], data.X_test, data.f_test, data.y_test) for c in CRITERIA}
method = "4. Hierarchical Decision Tree"
print_rows(
    [
        (method, c.capitalize(), train[c]["overall"], test[c]["overall"], test[c]["router"])
        for c in CRITERIA
    ]
)
print("Family %: the router's test accuracy.")
print()
print(f"{'test %':<10}{'router':>8}{'overall':>9}{'oracle':>8}{'p_A':>7}{'p_B':>7}")
for c in CRITERIA:
    t = test[c]
    values = (t["router"], t["overall"], t["oracle"], t["p_A"], t["p_B"])
    print(f"{c:<10}" + "".join(f"{100 * v:>{w}.2f}" for v, w in zip(values, (8, 9, 8, 7, 7))))

model = hierarchy["gini"]
families, species = np.unique(data.f_train), np.unique(data.y_train)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), gridspec_kw={"width_ratios": [1, 2]})
routed = model.predict_family(data.X_test)
heatmap(axes[0], confusion_matrix(data.f_test, routed, families), families, "Router, Gini, test")
predicted = model.predict(data.X_test)
heatmap(
    axes[1], confusion_matrix(data.y_test, predicted, species), species, "Final species, Gini, test"
)
plt.tight_layout()
plt.show()

The same experts behind a weaker router (max_depth = 4): routing errors pass straight through
to the species.

In [ ]:
weak_router = p1.DecisionTree("gini", max_depth=4).fit(X, data.f_train)
weak = p4.HierarchicalClassifier(weak_router, hierarchy["gini"].experts)
result = p4.analyse(weak, data.X_test, data.f_test, data.y_test)
print(", ".join(f"{k} {100 * v:.2f}" for k, v in result.items()))

## Part 5: Gaussian naive Bayes

Independent Gaussian features within each species, worked in log space. The MFCCs are far from
independent, and MFCC 1 never varies within Ameeregatrivittata in training.

In [ ]:
p5.sanity_check()
nb = p5.GaussianNB().fit(X, y)
print_rows([("5. Gaussian Naive Bayes", "-", *data.scores(nb))])

corr = np.corrcoef(X, rowvar=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), gridspec_kw={"width_ratios": [1, 1.25]})
im = axes[0].imshow(corr, cmap=DIVERGING, vmin=-1, vmax=1)
axes[0].set_xticks(range(22), range(1, 23), fontsize=7)
axes[0].set_yticks(range(22), range(1, 23), fontsize=7)
axes[0].set(xlabel="MFCC", ylabel="MFCC")
axes[0].set_title("Correlation of the 22 MFCCs, training rows", loc="left")
axes[0].grid(False)
fig.colorbar(im, ax=axes[0], shrink=0.75, label="r")
heatmap(
    axes[1],
    confusion_matrix(data.y_test, nb.predict(data.X_test), nb.classes_),
    nb.classes_,
    "Naive Bayes, test",
)
plt.tight_layout()
plt.show()

The Ameeregatrivittata column above collects rows from other species: MFCC 1 is 1.0 in every
training row of that species, so its variance there is only epsilon. Without MFCC 1 (an
experiment, not part of the matrix) the effect goes away:

In [ ]:
k = list(nb.classes_).index("Ameeregatrivittata")
print(
    f"variance of MFCC 1 within Ameeregatrivittata: {nb.var_[k, 0]:.3e} (epsilon {nb.epsilon_:.3e})"
)
without = p5.GaussianNB().fit(X[:, 1:], y)
test = 100 * accuracy(data.y_test, without.predict(data.X_test[:, 1:]))
print(f"test accuracy without MFCC 1: {test:.2f}%")